# 01 · Pre-processing (Methods, D-BIAT Data Pre-processing)

**Input:** `DBIAT_data_N135.csv`, the 120 D-BIAT trials of each of the 135 analyzed participants
(Control 45, Depressed 43, Suicidal 47; the columns are described in the README).

**Rules** (`settings.yaml → preprocessing`), as described in the manuscript:
1. The last (20th) trial of every block is removed, which leaves 114 trials per participant.
2. A participant is excluded if more than 10% of the remaining trials are faster than 300 ms, or if accuracy is below 30%.
   All 135 participants in the data file meet these criteria; the check is repeated here.
3. Trials with RT < 400 ms or > 2000 ms are removed.

**Output:** the retained trials (`outputs/derived/trials_clean.csv`) and the trial-exclusion summary.

In [ ]:
import sys
sys.path.insert(0, "..")          # dbiat_analysis.py is in the folder above notebooks/
import pandas as pd
from scipy import stats
import dbiat_analysis as A

cfg = A.load_settings()
PP = cfg["preprocessing"]
data = A.load_data(cfg)
n = data.groupby("group").participant.nunique().reindex(A.GROUPS)
print(data.participant.nunique(), "participants;", len(data), "trials;", n.to_dict())
assert data.participant.nunique() == 135 and n.tolist() == [45, 43, 47]
assert (data.groupby("participant").size() == 120).all()

## 1. Apply the rules

In [ ]:
prep = A.preprocess(data, PP)
part = prep.participants
assert part.meets_criteria.all() and (part.n_trials == 114).all()
print(f"Largest proportion of RTs < {PP['subject_fast_rt_ms']} ms: {part.prop_fast.max():.3f} "
      f"(criterion: more than {PP['subject_fast_rt_max_prop']:.2f})")
print(f"Lowest accuracy: {part.accuracy.min():.3f} (criterion: below {PP['subject_min_accuracy']:.2f})")
low = part[part.accuracy < 0.70]
print("Participants with accuracy below 70% (the stricter criterion):", len(low), low.group.value_counts().to_dict())

## 2. Trials outside 400–2000 ms, overall and by group
The Kruskal–Wallis test compares the per-participant percentages of excluded trials across groups.

In [ ]:
rows = [dict(sample="All (N = 135)", trials=int(part.n_trials.sum()), excluded=int(part.n_outside_window.sum()))]
for g in A.GROUPS:
    d = part[part.group == g]
    rows.append(dict(sample=g, trials=int(d.n_trials.sum()), excluded=int(d.n_outside_window.sum())))
te = pd.DataFrame(rows)
te["percent"] = (100 * te.excluded / te.trials).round(2)
H, p = stats.kruskal(*[part.loc[part.group == g, "pct_outside_window"] for g in A.GROUPS])
te["kruskal_wallis_p"] = [p] + [None] * 3
A.write_table(te, cfg, "trial_exclusions")
print(f"Kruskal–Wallis on per-participant percentages: H = {H:.2f}, {A.p_eq(p)}")
te

## 3. Save the retained trials

In [ ]:
prep.trials.to_csv(A.path(cfg, "derived", "trials_clean.csv"), index=False)
print(len(prep.trials), "retained trials")